# 03_train_hog_svm.ipynb

Baseline truyền thống HOG + SVM trên DurianLDD. Chạy trên CPU, cùng split khóa với các baseline học sâu, báo macro F1 trên validation. Không tối ưu siêu tham số.

Cell môi trường kéo nhánh `GIT_BRANCH` và gắn ảnh từ `cthng123/durian-leaf-disease-dataset`. Split lấy từ `manifests/split_manifest.csv`. Tập test không mở.

## 1. Môi trường

Internet để On. Accelerator None. Dataset input là `cthng123/durian-leaf-disease-dataset`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "main"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

## 2. Dữ liệu và cấu hình

Ảnh xám 224×224, không augmentation. Tập train và validation đọc từ cột `split` của manifest.

In [ ]:
from src.training.run import load_configs

CONFIG = "configs/train_hog_svm.yaml"
train_cfg, pipeline_cfg = load_configs(CONFIG)

print(f"Seed: {train_cfg['seed']}")
print(f"HOG: {train_cfg['hog']}")
print(f"SVM: {train_cfg['svm']}")
print("Thiết bị: cpu")
print("Tập test: không mở")

## 3. Trích đặc trưng, train và kết quả validation

Trích HOG cho train và validation, chuẩn hóa, fit SVM, rồi tính macro F1, balanced accuracy, recall từng lớp và confusion matrix. Mất khoảng bốn phút trên CPU.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from src.training.hog_svm import main

paths = main(CONFIG)
record = pd.read_json(paths["metrics"], typ="series")
display(pd.DataFrame([{
    "val_macro_f1": record["val_macro_f1"],
    "val_balanced_accuracy": record["val_balanced_accuracy"],
    "val_accuracy": record["val_accuracy"],
    "feature_dimension": record["feature_dimension"],
    "seconds": record["elapsed_seconds"],
}]))
display(pd.DataFrame(
    [{"class_name": name, "recall": value} for name, value in record["per_class_recall"].items()]
))
display(Image(filename=paths["confusion_matrix"]))